# Etapa 2 — Preparação da Base: Bank Marketing

Continuação da EDA (`00_eda.ipynb`). Objetivo desta etapa:
1. Aplicar as transformações sem estado decididas na EDA (remover vazamento, criar o target binário, tratar o código sentinela de `pdays`).
2. Fixar a fronteira entre **contexto** (features do cliente) e **braço** (ação controlável pela instituição) para o desenho do bandit.
3. Ajustar (`fit`) a codificação das categóricas de contexto e persistir os artefatos que as Etapas 3, 4 e 5 vão reaproveitar.

Toda a lógica reutilizável vive em `src/tech_challenge_5/features.py` — este notebook documenta as decisões e chama essas funções, em vez de duplicar a transformação em células soltas.

In [1]:
import pandas as pd
from pathlib import Path

from tech_challenge_5.features import (
    ARM_COL,
    CATEGORICAL_CONTEXT_COLS,
    CONTEXT_COLS,
    ContextEncoder,
    NUMERIC_CONTEXT_COLS,
    TARGET_BIN_COL,
    build_dataset,
    clean_data,
    load_raw,
)

RAW_PATH = Path("../data/raw/bank-additional-full.csv")
PROCESSED_DIR = Path("../data/processed")

## 1. Limpeza (transformações sem estado)

Reaplica exatamente as decisões da EDA (Etapa 1):
- remove `duration` (vazamento temporal);
- cria `y_bin` a partir de `y`;
- transforma `pdays == 999` na flag `nunca_contatado` em vez de número contínuo;
- mantém `"unknown"` como categoria própria (nenhuma linha é descartada).

In [2]:
df_raw = load_raw(RAW_PATH)
df = clean_data(df_raw)
print("raw:", df_raw.shape, "-> limpo:", df.shape)
df.head(3)

raw: (41188, 21) -> limpo: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,y_bin,nunca_contatado
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,0,1


## 2. Contexto, braço e alvo

**Decisão de braço**: entre os candidatos levantados na EDA (`contact` e/ou `month`), fixamos `contact` (`cellular` vs `telephone`) como o braço do bandit:
- é binário — mantém o espaço de ação simples;
- é uma decisão de canal que a instituição controla diretamente por contato;
- evita diluir as poucas conversões (dataset desbalanceado) em muitos braços, como aconteceria com `month` (10 categorias).

`month` e `poutcome` permanecem como **contexto** (sinal temporal/histórico), não como ação testável. `job`, `education`, `marital` etc. são atributos fixos do cliente — contexto por definição.

In [3]:
context, arm, target = df[CONTEXT_COLS], df[ARM_COL], df[TARGET_BIN_COL]

print("Contexto:", context.shape)
print("Colunas numéricas:", NUMERIC_CONTEXT_COLS)
print("Colunas categóricas:", CATEGORICAL_CONTEXT_COLS)
print("Braço (arm):", ARM_COL, "->", sorted(arm.unique()))
context.head(3)

Contexto: (41188, 17)
Colunas numéricas: ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Colunas categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'month', 'poutcome']
Braço (arm): contact -> ['cellular', 'telephone']


,age,campaign,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,job,marital,education,default,housing,loan,month,poutcome,nunca_contatado
0,56,1,0,1.1,93.994,-36.4,4.857,5191.0,housemaid,married,basic.4y,no,no,no,may,nonexistent,1
1,57,1,0,1.1,93.994,-36.4,4.857,5191.0,services,married,high.school,unknown,no,no,may,nonexistent,1
2,37,1,0,1.1,93.994,-36.4,4.857,5191.0,services,married,high.school,no,yes,no,may,nonexistent,1


## 3. Conversão observada por braço

Checagem rápida — a Etapa 3 (baseline + bandit) vai usar taxas como esta para comparar a política fixa contra a adaptativa.

In [4]:
conversion_by_arm = (
    pd.DataFrame({ARM_COL: arm, TARGET_BIN_COL: target})
    .groupby(ARM_COL)[TARGET_BIN_COL]
    .agg(["mean", "count"])
    .rename(columns={"mean": "taxa_conversao"})
)
conversion_by_arm

,taxa_conversao,count
contact,,
cellular,0.147376,26144
telephone,0.052313,15044


## 4. Codificação do contexto (`ContextEncoder`)

Diferente da limpeza acima, o one-hot das categóricas é uma transformação **com estado**: precisa ser ajustada (`fit`) uma única vez sobre a base de treino e depois reaplicada — em lote aqui, e por cliente nas Etapas 4 e 5 — sem recalcular categorias do zero a cada vez. Isso é o que evita *train/serving skew* entre o que o modelo aprendeu e o que ele recebe em produção.

`OneHotEncoder(handle_unknown="ignore")` garante que uma categoria nunca vista em produção não quebre a transformação (a linha correspondente sai zerada naquele grupo de colunas, em vez de gerar erro).

In [5]:
encoder = ContextEncoder.fit(context)
encoded_sample = encoder.transform(context.head(3))
print("shape codificado:", encoded_sample.shape)
encoded_sample.iloc[:, :8]

shape codificado: (3, 55)


,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Testando o caso de uma categoria nunca vista (simulando um cliente novo em produção):

In [6]:
cliente_categoria_nova = context.iloc[[0]].copy()
cliente_categoria_nova["job"] = "categoria-nunca-vista"
encoded_novo = encoder.transform(cliente_categoria_nova)
print("transformou sem erro, shape:", encoded_novo.shape)

transformou sem erro, shape: (1, 55)


## 5. Persistir artefatos

Dois artefatos saem desta etapa, ambos em `data/processed/` (gitignored — são regeneráveis a partir do raw + deste notebook, não versionamos dados):

- `bank_marketing_prepared.csv`: contexto (colunas originais, sem one-hot) + `contact` (braço) + `y_bin` (alvo) — formato legível, usado pela Etapa 3 para calcular baseline e treinar/simular o bandit.
- `context_encoder.joblib`: o `ColumnTransformer` já ajustado — reaproveitado nas Etapas 4 (golden set) e 5 (serviço) para transformar um cliente novo exatamente como o treino foi transformado.

In [7]:
prepared = context.copy()
prepared[ARM_COL] = arm
prepared[TARGET_BIN_COL] = target

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
prepared.to_csv(PROCESSED_DIR / "bank_marketing_prepared.csv", index=False)
encoder.save(PROCESSED_DIR / "context_encoder.joblib")

print("Salvo em:", PROCESSED_DIR.resolve())
print(prepared.shape)

Salvo em: C:\Users\lfell\pyprojects\fiap\mlengineering\tech_challenge_5\data\processed
(41188, 19)


## 6. Conclusões — o que fica pronto para a Etapa 3

- **Contexto** (17 colunas: numéricas + categóricas + `nunca_contatado`), **braço** (`contact`, 2 valores) e **alvo** (`y_bin`) estão definidos e separados.
- A limpeza (sem estado) e a codificação (com estado, via `ContextEncoder`) vivem em `src/tech_challenge_5/features.py`, prontas para serem reaproveitadas pela Etapa 3 (lendo o CSV preparado diretamente, sem recodificar) e pelas Etapas 4/5 (chamando `ContextEncoder.load(...)` para transformar um cliente novo por vez).
- Taxa de conversão por braço calculada acima já dá o primeiro número de referência para o baseline determinístico da Etapa 3.